# 21 - Offline DQN with an Auxiliary Latent Cosine Loss

Use example 02’s Hub replay, augmentation, tokenizer, packed episodes, and pretrained
backbone. The encoder contains only the embedding and backbone. The Q head and
transition model are separate modules; there is no projection.

`z = encoder(inputs)["z"]`

`q = q_head(z)`

`predicted_next_z = transition(z, action)`

Train with `L = L_dqn + COSINE_LOSS_WEIGHT * L_cosine`. The transition uses a separate
learned action embedding and residual feed-forward blocks. The cosine target comes
from the frozen delayed encoder. This is a usage example, not a full experiment.


In [ ]:
import json
from copy import deepcopy
from pathlib import Path

import torch
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    environment_episode_ranges,
    Augmenter,
    DataLoader,
    Tokenizer,
    compose,
    load_stores_from_hub,
    to_device,
)
from mouse_core.models import (
    Model,
    DecodeCache,
    prepare_decode,
    Backbone,
    Embedding,
    MultiEmbedding,
    MLP,
)
from mouse_core.objectives import CosineObjective, DqnObjective, boundary_discount

# Artifacts
CHECKPOINT_PATH = Path("runs/offline-dqn-cosine.pt")

# Repositories
DATASET_ID = "mouse-example-frozenlake-dataset"  # Hugging Face dataset repo for load_stores_from_hub

# Environment
MAX_ACTIONS = 4  # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64  # vocabulary size for discrete observations
MAX_STEPS_PER_EPISODE = 24  # max steps per episode (matches collect)
REPEAT_COUNT = 10  # inner episodes per outer task (matches collect)

# Model
PRETRAINED = "Qwen/Qwen3-0.6B"  # HF checkpoint for Tokenizer, Embedding, and Backbone
TRANSITION_BLOCKS = 2
HEAD_LAYERS = 1
HEAD_SCALE = 0.1  # final prediction linear's initialization

# Training
TOKEN_BUDGET = 16384  # packed tokens; whole episodes are added until the next one would not fit
NUM_CYCLES = 2  # outer train cycles (print cadence)
TRAIN_STEPS = 50  # optimizer updates per cycle (passed to run_train)

# Auxiliary objective
COSINE_LOSS_WEIGHT = 0.1
COSINE_EPS = 1e-8

# Optimizer learning rates
LR_EMBEDDING = 1e-5  # token embeddings and numeric Fourier features
LR_BACKBONE = 1e-5  # backbone
LR_TRANSITION = 1e-5
LR_HEADS = 1e-5  # prediction heads and their input RMSNorms

# Target interpolation rates
POLYAK_TAU_EMBEDDING = 0.1  # delayed embedding
POLYAK_TAU_BACKBONE = 0.1  # delayed backbone
POLYAK_TAU_VALUE = 0.001  # delayed value head

# Runtime
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)

## Data Pipeline

Use the event pipeline described in [02 — Offline DQN](02_train_offline_dqn.ipynb):
`source events → sparse decisions → augmenter → tokenizer → pack → embedding`.
Packing derives `action_executed` from the presence of an agent event. Cosine prediction
uses that mask to exclude reset frames, which have no incoming action.

Compose `train_transform = compose(stages=(augmenter.events, tokenizer.encode))`
and pass `train_transform` to the loader. The augmenter reads event payloads directly. Each sampled outer task
gets a separate `group_id`, retaining its repeated lake episodes together while isolating
attention and objective transitions from other tasks. `TOKEN_BUDGET` counts packed tokens,
including group prefixes; the loader keeps whole samples and rejects an oversized first sample.

For evaluation, adapt the encoder and Q-head inference pattern from
[example 15](15_inference.ipynb), keeping this notebook's model and tokenizer settings.
Omit training augmentation so selected actions match the environment; the auxiliary
transition model is used only for training.


In [ ]:
def on_group_start(ctx):
    return bool(ctx["group_start"])


def on_inner_episode_ended(ctx):
    return bool(ctx.get("inner_terminated", False) or ctx.get("inner_truncated", False))


group_prefix = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\n"
)

augmenter = Augmenter(
    seed_field="episode_index",
    fields=[
        {
            "type": "discrete",
            "provider": None,
            "input_field": "action",
            "input_vector_field": "q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True},
        {
            "type": "discrete",
            "provider": None,
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            'type': 'text',
            "tokenizer": "text",
            'provider': None,
            'format': group_prefix,
            'when': on_group_start,
        },
        {
            'type': 'text',
            "tokenizer": "text",
            'provider': 'environment',
            'input_field': 'repeat_episode_index',
            'format': 'episode {field+1}\n',
        },
        {
            'type': 'text',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'action',
            'lookup': ['a', 'b', 'c', 'd'],
            'format': '{field}',
        },
        {
            'type': 'text',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'observation',
            'format': ',{field}',
        },
        {
            'type': 'text',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'reward',
            'format': ',r={field:g}',
        },
        {
            'type': 'text',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'inner_terminated',
            'format': ',terminated',
        },
        {
            'type': 'text',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'inner_truncated',
            'format': ',truncated',
        },
        {
            'type': 'token',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'action_requested',
            'format': '\n',
            'head_output': True,
        },
        {
            'type': 'text',
            "tokenizer": "text",
            'provider': 'environment',
            'when': on_inner_episode_ended,
            'format': '\n\n',
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "truncated",
            "missing_value": False,
        },
    ],
)

train_transform = compose(stages=(augmenter.events, tokenizer.encode))


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    batch_size=1,
    sample_ranges=environment_episode_ranges,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)

## Build Encoder, Q Head, and Transition Model

`Encoder` runs `Embedding → Backbone → select readout tokens`. It returns the raw
backbone readout as `out["z"]`, along with cache and token-position metadata. There
is no projection, Q head, action selection, or transition inside the encoder.

`QHead` builds its MLP internally. The standalone `q_head(z)` retains its normal input RMSNorm and predicts one Q-value
per action. `transition(z, action)` has its own learned action embedding table and
full-width additive conditioning: `h = RMSNorm(z) + action_embedding(action)`,
then prenormalized residual blocks: `h = h + MLP(h)`.
The action embedding is independent of the encoder’s token embeddings.

Both losses train the encoder. DQN trains the Q head; cosine matching trains the
transition. Targets come from `delayed_encoder`, a frozen copy. Separate frozen
`backup_value_head` tracks the Q head for backup values. The online Q head selects backup actions from the delayed encoder readouts.
The existing cached-encoding path is retained. For inference, apply `q_head` to the
encoder output, then choose an action from those Q-values.


In [ ]:
class TransitionModel(nn.Module):
    def __init__(self, *, latent_dim, num_actions, num_blocks):
        super().__init__()
        self.input_norm = nn.RMSNorm(latent_dim, eps=1e-5)
        self.action_embedding = nn.Embedding(num_actions, latent_dim)
        self.blocks = nn.ModuleList([
            MLP(
                in_features=latent_dim,
                out_features=latent_dim, hidden_dim=4 * latent_dim,
                num_layers=2,
            )
            for _ in range(num_blocks)
        ])
        self.output_norm = nn.RMSNorm(latent_dim, eps=1e-5)

    def forward(self, z, action):
        z = self.input_norm(z.float()) + self.action_embedding(action)
        # Each MLP prenormalizes its input; the skip path retains the latent state.
        for block in self.blocks:
            z = z + block(z)
        return self.output_norm(z)

class QHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)

class Encoder(Model):
    def __init__(self, *, pretrained, tokenizer_sources, train_kernel, decode_kernel, dtype):
        super().__init__()
        self.embedding = MultiEmbedding(embeddings={
            name: Embedding(pretrained=source) for name, source in tokenizer_sources.items()
        })
        self.backbone = Backbone(
            pretrained=pretrained,
            train_kernel=train_kernel,
            decode_kernel=decode_kernel,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to_tensors(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_ids"]
        head_step_ids = tensors["head_step_ids"]
        embed = self.embedding(
            token_ids, numeric_values=numeric_values, numeric_mask=numeric_mask,
            tokenizer_ids=tensors["tokenizer_ids"], tokenizer_names=tokens.tokenizer_names,
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed, tokens=tokens, sequence_ids=sequence_ids,
                head_token_ids=head_token_ids, head_step_ids=head_step_ids,
            )
            if cache is None:
                cache = DecodeCache(sessions=(self.backbone.decode_session(batch_size=tokens.B),))
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape, features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }

encoder = Encoder(
    tokenizer_sources=tokenizer.tokenizer_sources,
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    pretrained=PRETRAINED,
).train().to(device)

q_head = QHead(
    hidden_dim=encoder.backbone.hidden_dim,
    num_actions=MAX_ACTIONS,
    num_layers=HEAD_LAYERS,
    scale=HEAD_SCALE,
).train().to(device)
transition = TransitionModel(
    latent_dim=encoder.backbone.hidden_dim, num_actions=MAX_ACTIONS,
    num_blocks=TRANSITION_BLOCKS,
).train().to(device) if COSINE_LOSS_WEIGHT > 0 else None

# Frozen target encoder and independent backup heads.
delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()

print(encoder)
print(q_head)
print(transition)


## Train

The loader and DQN configuration match example 02. A single optimizer owns separate
parameter groups for encoder embeddings, backbone, Q head, and transition model.
After each combined update, the delayed encoder and two backup heads receive their
respective Polyak updates.

Cosine loss compares `transition(z_t, action_next)` directly with the detached delayed
encoding at `t+1`. Actions are incoming fields: row `t+1` stores the action taken at
`t`, including the same permutation used in the tokenizer and DQN. Pairs cannot cross
packed groups, outer episode ends, or frames without an executed action (including inner resets).
Transitions into terminal observations remain valid; subsequent resets are excluded.
A batch without valid pairs contributes only DQN. `COSINE_LOSS_WEIGHT=0` disables the
auxiliary path, including transition construction and optimizer parameters.
Each residual block is a two-layer MLP with 4× expansion and no output scaling;
the transition ends with RMSNorm. Encoder readouts remain raw.
No SSL projection or extra prediction head is used.

`environment_episode_ranges` selects complete outer tasks from the event log. Unfinished tails remain in storage. `cross_group_backups="fault"` raises if a training target needs a value beyond its sampled group; zero discount at outer termination or truncation ends the return.


In [ ]:
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(lr={
            "embedding": LR_EMBEDDING,
            "backbone": LR_BACKBONE,
        }),
        {"params": q_head.parameters(), "lr": LR_HEADS},
        *([{ "params": transition.parameters(), "lr": LR_TRANSITION }]
          if transition is not None else []),
    ],
    lr=LR_BACKBONE,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-8,
)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
    temperature=0.0,
    cross_group_backups="fault",
    gate=None,
)
cosine_objective = CosineObjective(
    mask_key=None,
    eps=COSINE_EPS
) if COSINE_LOSS_WEIGHT > 0 else None

def run_train(*, num_steps: int) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    encoder.train()
    q_head.train()
    if transition is not None:
        transition.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data, group_id = loader.next_batch()

        out = encoder(inputs)
        with torch.no_grad():
            delayed_out = delayed_encoder(inputs)
            backup_value = backup_value_head(delayed_out["z"])
            backup_action = q_head(delayed_out["z"])
        objective_data = to_device(data=objective_data, device=device)
        group_id = group_id.to(device)
        losses, metrics = objective(
            objective_data=objective_data,
            group_id=group_id,
            predictions=q_head(out["z"]),
            backup_value_predictions=backup_value,
            backup_action_predictions=backup_action,
        )
        loss = torch.stack(tuple(losses.values())).sum()

        if cosine_objective is not None:
            assert transition is not None
            auxiliary_losses, auxiliary_metrics = cosine_objective.latent_cosine_loss(
                transition=transition, predictions=out["z"], targets=delayed_out["z"],
                reset_key=None,
                objective_data=objective_data, group_id=group_id,
            )
            loss = loss + COSINE_LOSS_WEIGHT * auxiliary_losses["cosine"]
            metrics.update(auxiliary_metrics)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        polyak_update(target=delayed_encoder.embedding, source=encoder.embedding, tau=POLYAK_TAU_EMBEDDING)
        polyak_update(target=delayed_encoder.backbone, source=encoder.backbone, tau=POLYAK_TAU_BACKBONE)
        polyak_update(target=backup_value_head, source=q_head, tau=POLYAK_TAU_VALUE)
    assert loss is not None
    return loss, metrics

## Run

Each cycle performs `TRAIN_STEPS` combined optimizer updates.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(num_steps=TRAIN_STEPS)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")

loader.close()

## Save Locally

Save the three online modules separately. Reconstruct the encoder, Q head, and
transition with the same settings before loading their state dictionaries.
This checkpoint contains weights, not optimizer or delayed-network state.


In [ ]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "encoder": encoder.state_dict(),
    "q_head": q_head.state_dict(),
    "transition": None if transition is None else transition.state_dict(),
}, CHECKPOINT_PATH)
print(f"Saved encoder, Q head, and transition to {CHECKPOINT_PATH}")
